In [1]:
from google.colab import files
uploaded = files.upload()  # opens a file picker

Saving XD.zip to XD.zip


In [3]:
import zipfile
with zipfile.ZipFile("XD.zip", "r") as z:
    z.extractall("/content/gender-class/")

In [4]:
import cv2, os, glob, shutil
import numpy as np
from skimage import transform, filters

BASE_INPUT_PATH = "/content/gender-class/"
SUB_FOLDERS = [
    "Training/female", "Training/male",
    "Validation/female", "Validation/male",

]
OUT_PREPROCESSED = "/content/stage1_preprocessed/"
OUT_FILTERED     = "/content/stage2_filtered/"
OUT_EDGE_SEG     = "/content/stage3_edge_seg/"
OUT_ZIP          = "/content/final_results"

RESIZE_DIM   = (128, 128)
GAUSS_SIGMA  = 1
GAUSS_KERNEL = (5, 5)
MEDIAN_K     = 5
SHARP_KERNEL = np.array([[-1,-1,-1], [-1, 9,-1], [-1,-1,-1]])
SAMPLE_SIZE  = 1000
CANNY_T1, CANNY_T2 = 100, 200
THRESH_VALUE = 127

def ensure_dir(path): os.makedirs(path, exist_ok=True)


def stage1_preprocess():

    print("=== STAGE 1 Preprocessing ===")
    for sub in SUB_FOLDERS:
        out_sub = os.path.join(OUT_PREPROCESSED, sub)
        ensure_dir(out_sub)
        files = glob.glob(os.path.join(BASE_INPUT_PATH, sub, "*"))
        print(f"  [{sub}] {len(files)} images")
        skipped = 0
        for fp in files:
            img = cv2.imread(fp, 0)
            if img is None: skipped += 1; continue
            img_r = transform.resize(img, RESIZE_DIM)
            img_e = filters.gaussian(img_r, sigma=GAUSS_SIGMA)
            final = (img_e * 255).astype(np.uint8)
            cv2.imwrite(os.path.join(out_sub, os.path.basename(fp)), final)
        print(f"     done  (skipped {skipped} unreadable)")
    print("Stage 1 complete.\n")


def stage2_filter():

    print("=== STAGE 2 Image Filtering ===")
    total = 0
    for root, _, files in os.walk(OUT_PREPROCESSED):
        for fname in files:
            if not fname.lower().endswith(('.png','.jpg','.jpeg')): continue
            img = cv2.imread(os.path.join(root, fname))
            if img is None: continue
            rel = os.path.relpath(root, OUT_PREPROCESSED)
            save_dir = os.path.join(OUT_FILTERED, rel)
            ensure_dir(save_dir)
            _gaussian  = cv2.GaussianBlur(img, GAUSS_KERNEL, 0)
            _median    = cv2.medianBlur(img, MEDIAN_K)
            sharpened  = cv2.filter2D(img, -1, SHARP_KERNEL)
            cv2.imwrite(os.path.join(save_dir, "filtered_" + fname), sharpened)
            total += 1
    print(f"Stage 2 complete — {total} images filtered.\n")


def stage3_edge_seg():

    print("=== STAGE 3 — Edge Detection & Segmentation ===")
    if os.path.exists(OUT_EDGE_SEG): shutil.rmtree(OUT_EDGE_SEG)
    ensure_dir(OUT_EDGE_SEG)
    imgs = (glob.glob(os.path.join(OUT_FILTERED,"**","*.jpg"),  recursive=True) +
            glob.glob(os.path.join(OUT_FILTERED,"**","*.jpeg"),recursive=True) +
            glob.glob(os.path.join(OUT_FILTERED,"**","*.png"),  recursive=True))
    sample = imgs[:SAMPLE_SIZE]
    processed = 0
    for path in sample:
        img  = cv2.imread(path, 0)
        if img is None: continue
        base = os.path.splitext(os.path.basename(path))[0]
        sx   = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
        sy   = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
        sobel = cv2.convertScaleAbs(cv2.magnitude(sx, sy))
        canny = cv2.Canny(img, CANNY_T1, CANNY_T2)
        _, seg = cv2.threshold(img, THRESH_VALUE, 255, cv2.THRESH_BINARY)
        cv2.imwrite(f"{OUT_EDGE_SEG}{base}_sobel.jpg",  sobel)
        cv2.imwrite(f"{OUT_EDGE_SEG}{base}_canny.jpg",  canny)
        cv2.imwrite(f"{OUT_EDGE_SEG}{base}_seg.jpg",    seg)
        processed += 1
    zip_path = shutil.make_archive(OUT_ZIP, "zip", OUT_EDGE_SEG)
    print(f"Stage 3 complete — {processed} images. ZIP: {zip_path}\n")


if __name__ == "__main__":
    stage1_preprocess()
    stage2_filter()
    stage3_edge_seg()
    print("ALL DONE")
    print(f"  Preprocessed : {OUT_PREPROCESSED}")
    print(f"  Filtered     : {OUT_FILTERED}")
    print(f"  Edge / Seg   : {OUT_EDGE_SEG}")
    print(f"  ZIP          : {OUT_ZIP}.zip")


=== STAGE 1 Preprocessing ===
  [Training/female] 23243 images
     done  (skipped 0 unreadable)
  [Training/male] 23766 images
     done  (skipped 0 unreadable)
  [Validation/female] 5841 images
     done  (skipped 0 unreadable)
  [Validation/male] 5808 images
     done  (skipped 0 unreadable)
Stage 1 complete.

=== STAGE 2 Image Filtering ===
Stage 2 complete — 58658 images filtered.

=== STAGE 3 — Edge Detection & Segmentation ===
Stage 3 complete — 1000 images. ZIP: /kaggle/working/final_results.zip

ALL DONE
  Preprocessed : /kaggle/working/stage1_preprocessed/
  Filtered     : /kaggle/working/stage2_filtered/
  Edge / Seg   : /kaggle/working/stage3_edge_seg/
  ZIP          : /kaggle/working/final_results.zip


In [6]:
from google.colab import files

# Download the ZIP (Stage 3 results)
files.download('/kaggle/working/final_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>